# Test du Thm 3.1 — Intégration de chemin conservative

Ce notebook reprend le script [test_path_independence.py](test_path_independence.py) et vérifie que l'intégration de chemin est conservative pour trois circuits fermés.

Exécute les cellules dans l'ordre.

In [2]:
from pathlib import Path
import math
import sys
from typing import List, Tuple

import torch

# Permet l'import depuis le notebook même si le kernel démarre ailleurs.
project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "cortical_column.py").exists():
        project_root = candidate
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from cortical_column import GridCellNetwork

In [3]:
# ─── Utilitaires ─────────────────────────────────────────────────────────────

def torus_dist(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Distance L2 entre deux points sur le tore [0,1)^D."""
    diff = torch.abs(a - b)
    wrapped_diff = torch.min(diff, 1.0 - diff)
    return torch.norm(wrapped_diff)


def run_circuit(
    grid_net: GridCellNetwork,
    velocities: List[torch.Tensor],
    dt: float = 1.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Exécute un circuit de déplacements et retourne l'état initial, final et l'erreur."""
    L_0 = grid_net.get_location_state().clone()

    for v in velocities:
        grid_net.integrate_all(v, dt=dt)

    L_final = grid_net.get_location_state().clone()
    err = torus_dist(L_final, L_0)

    return L_0, L_final, err

In [5]:
# ─── Définition des circuits ──────────────────────────────────────────────────

def make_square_circuit(d: int = 2) -> List[torch.Tensor]:
    """Circuit carré : 4 pas qui se compensent exactement."""
    assert d >= 2, "Le circuit carré requiert d >= 2"

    def vec(x: float, y: float) -> torch.Tensor:
        v = torch.zeros(d, dtype=torch.float32)
        v[0] = x
        v[1] = y
        return v

    return [
        vec(0.0, 1.0),
        vec(1.0, 0.0),
        vec(0.0, -1.0),
        vec(-1.0, 0.0),
    ]


def make_triangle_circuit(d: int = 2) -> List[torch.Tensor]:
    """Circuit triangulaire équilatéral : 3 pas dont la somme est zéro."""
    assert d >= 2, "Le circuit triangulaire requiert d >= 2"

    cos60 = 0.5
    sin60 = math.sqrt(3) / 2

    def vec(x: float, y: float) -> torch.Tensor:
        v = torch.zeros(d, dtype=torch.float32)
        v[0] = x
        v[1] = y
        return v

    return [
        vec(1.0, 0.0),
        vec(-cos60, sin60),
        vec(-cos60, -sin60),
    ]


def make_random_circuit(N: int = 20, d: int = 2, seed: int = 42) -> List[torch.Tensor]:
    """Circuit aléatoire : N-1 pas aléatoires + 1 pas de fermeture exact."""
    assert N >= 2, "Un circuit requiert au moins 2 pas"

    torch.manual_seed(seed)
    velocities = []

    for _ in range(N - 1):
        v = torch.rand(d, dtype=torch.float32) * 2 - 1
        velocities.append(v)

    v_closing = -torch.stack(velocities).sum(dim=0)
    velocities.append(v_closing)

    total = torch.stack(velocities).sum(dim=0)
    assert torch.allclose(total, torch.zeros(d), atol=1e-5), f"Erreur de construction du circuit aléatoire : somme = {total}"

    return velocities

In [6]:
# ─── Configuration et affichage ───────────────────────────────────────────────

M: int = 8
d: int = 8
TOLERANCE: float = 1e-4

print("=" * 60)
print("Test Thm 3.1 — Intégration de chemin conservative")
print(f"Configuration : M={M} modules, d={d} dim/module (D_total={M * d})")
print("=" * 60)

grid_net = GridCellNetwork(K=M, d=d)


def run_and_report(case_name: str, velocities: List[torch.Tensor]):
    L_0, L_final, err = run_circuit(grid_net, velocities)

    print(f"      L_0     (extrait) : {L_0[:4].tolist()}")
    print(f"      L_final (extrait) : {L_final[:4].tolist()}")
    print(f"      Erreur de fermeture (distance torique L2) : {err.item():.2e}")

    if err >= TOLERANCE:
        print(f"  !! ECHEC : err={err.item():.6f} >= tolerance={TOLERANCE}")
        print(f"     L_0     = {L_0.tolist()}")
        print(f"     L_final = {L_final.tolist()}")
        print(f"     diff    = {(L_final - L_0).tolist()}")
    else:
        print(f"      OK — err < {TOLERANCE}")

    assert err < TOLERANCE, f"{case_name} : erreur de fermeture {err.item():.6f} >= {TOLERANCE}"
    return L_0, L_final, err

Test Thm 3.1 — Intégration de chemin conservative
Configuration : M=8 modules, d=8 dim/module (D_total=64)


In [7]:
# ─── Circuit 1 : Carré ───────────────────────────────────────────────────────

print("\n[1/3] Circuit carré (4 pas, directions N-E-S-O)")

grid_net.reset_all()

square_vels = make_square_circuit(d=d)
print(f"      Vecteurs : {[v[:2].tolist() for v in square_vels]}  (2 premières composantes)")

L0_sq, Lf_sq, err_sq = run_and_report("Circuit carré", square_vels)


[1/3] Circuit carré (4 pas, directions N-E-S-O)
      Vecteurs : [[0.0, 1.0], [1.0, 0.0], [0.0, -1.0], [-1.0, 0.0]]  (2 premières composantes)
      L_0     (extrait) : [0.0, 0.0, 0.0, 0.0]
      L_final (extrait) : [0.0, 0.0, 0.0, 0.0]
      Erreur de fermeture (distance torique L2) : 0.00e+00
      OK — err < 0.0001


In [8]:
# ─── Circuit 2 : Triangle équilatéral ───────────────────────────────────────

print("\n[2/3] Circuit triangulaire équilatéral (3 pas)")

grid_net.reset_all()

triangle_vels = make_triangle_circuit(d=d)
print(f"      Vecteurs : {[v[:2].tolist() for v in triangle_vels]}  (2 premières composantes)")

L0_tr, Lf_tr, err_tr = run_and_report("Circuit triangulaire", triangle_vels)


[2/3] Circuit triangulaire équilatéral (3 pas)
      Vecteurs : [[1.0, 0.0], [-0.5, 0.8660253882408142], [-0.5, -0.8660253882408142]]  (2 premières composantes)
      L_0     (extrait) : [0.0, 0.0, 0.0, 0.0]
      L_final (extrait) : [0.0, 0.0, 0.0, 0.0]
      Erreur de fermeture (distance torique L2) : 0.00e+00
      OK — err < 0.0001


In [9]:
# ─── Circuit 3 : Aléatoire ───────────────────────────────────────────────────

print("\n[3/3] Circuit aléatoire (N=20 pas, seed=42)")

grid_net.reset_all()

random_vels = make_random_circuit(N=20, d=d, seed=42)
print(f"      Nombre de pas : {len(random_vels)}")
print(f"      Somme totale  : {torch.stack(random_vels).sum(dim=0)[:2].tolist()} ... (≈ zéro)")

L0_rnd, Lf_rnd, err_rnd = run_and_report("Circuit aléatoire", random_vels)


[3/3] Circuit aléatoire (N=20 pas, seed=42)
      Nombre de pas : 20
      Somme totale  : [0.0, 0.0] ... (≈ zéro)
      L_0     (extrait) : [0.0, 0.0, 0.0, 0.0]
      L_final (extrait) : [-0.0, 0.0, 1.1920928955078125e-07, 0.0]
      Erreur de fermeture (distance torique L2) : 6.03e-07
      OK — err < 0.0001


In [10]:
# ─── Récapitulatif ───────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("Récapitulatif des erreurs de fermeture (Thm 3.1)")
print(f"  Circuit carré       : {err_sq.item():.2e}")
print(f"  Circuit triangulaire : {err_tr.item():.2e}")
print(f"  Circuit aléatoire   : {err_rnd.item():.2e}")
print(f"  Tolérance           : {TOLERANCE:.2e}")
print("=" * 60)
print("Tous les tests sont passés. Thm 3.1 vérifié.")


Récapitulatif des erreurs de fermeture (Thm 3.1)
  Circuit carré       : 0.00e+00
  Circuit triangulaire : 0.00e+00
  Circuit aléatoire   : 6.03e-07
  Tolérance           : 1.00e-04
Tous les tests sont passés. Thm 3.1 vérifié.
